# Hotspots

The `concentration` findings describe one column at a time: a
dominant level, a near-constant. A combination of columns can
concentrate where neither column does — one `city` × `channel`
cell holding far more rows than its two marginals predict, in a
table where both marginals look unremarkable. The `hotspots`
detector reports those cells as `concentration` findings of type
`hotspot`.


## The independence baseline

The scan runs over the shared encoded matrix — nominal columns
level-coded, numeric columns in the encoder's quantile bins —
for every pair of eligible features, and for triples at the deep
budget. For each cell it compares the observed row count with
the count expected under independence: the product of the cell's
marginal shares times the number of scanned rows. A cell is
reported when its standardized residual clears a Bonferroni
threshold over all cells examined at family α = 0.01, and it
also clears absolute bars on expected count, ratio, observed
count, and share of scanned rows.

Default eligibility requires between 2 and 20 non-missing bins or levels.
Most continuous columns use 32 bins and therefore do not enter this scan
without fewer distinct bin boundaries. Pairs may contain at most 1,024
cells, triples at most 4,096; deep examines up to 400 eligible triples.

Counting examined cells matters. A wide pair space contains
thousands of cells; at an unadjusted bar, chance alone fills the
report with the luckiest of them. The Bonferroni denominator is
the number of cells actually counted, and the evidence states it.


In [ ]:
import random

import polars as pl

import onceover


def frame(planted: bool, seed: int) -> pl.DataFrame:
    rng = random.Random(seed)
    if planted:
        rows = [("a1", "b1")] * 1000
        rest_a = ["a2", "a3", "a4", "a5"]
        rest_b = ["b2", "b3", "b4", "b5"]
        rows += [(rng.choice(rest_a), rng.choice(rest_b)) for _ in range(9000)]
        rng.shuffle(rows)
        return pl.DataFrame({"city": [r[0] for r in rows],
                             "channel": [r[1] for r in rows]})
    a = ["a1"] * 1000 + ["a2", "a3", "a4", "a5"] * 2250
    b = ["b1"] * 1000 + ["b2", "b3", "b4", "b5"] * 2250
    rng.shuffle(a)
    rng.shuffle(b)
    return pl.DataFrame({"city": a, "channel": b})


for label, planted, seed in (("planted cell", True, 11),
                             ("same marginals, independent", False, 12)):
    report = onceover.profile(frame(planted, seed))
    hits = [f for f in report.findings if f.get("type") == "hotspot"]
    print(f"{label}: {len(hits)} hotspot findings")
    for f in hits:
        print("  ", f["claim"])
        print("   evidence:", f["evidence"][0]["metrics"])


## Reading the evidence

`cell` names the level (or bin) of each column in the
combination. `observed_rows` and `expected_rows` are counts over
the encoded row sample — the scan works on the sample the
encoder produced, and `rows_scanned` states its size.
`ratio` is observed over expected. `cells_examined` is the
Bonferroni denominator: every cell of every eligible combination
counted in the scan, reported so the bar can be re-derived.


## What the detector does not report

- **Single-column concentration.** A dominant level in one
  column is a `value_concentration` finding. Hotspots require a
  combination whose joint count exceeds what the marginals
  already explain — the independent frame in the example, with
  the same marginals as the planted frame, produces no finding.
- **Triples at the standard budget.** Three-column cells are
  scanned only at the deep budget; at standard the scan is pairs.
- **Combinations over the cell caps.** A pair or triple whose
  level combinations exceed the scan's cell caps is skipped
  rather than sampled down to fit.
